# Project: WW3 Configuration - Ayy, It's Wave!

This project is about setting up a WW3 configuration and any nuances we might want to consider.

Download this project by either:

1. Running the CrocoDash CLI command: `crocodash template --machine derecho --notebook crocodash.projects.ww3`
2. Copying the file from your CrocoDash checkout: `demos/crocodash/projects/ww3.ipynb`

Grid Generation - The first thing we are going to do is think about how to get a WW3 domain going - do we want OBCs? How expensive is this domain going to be? 

For WW3, we need to think a bit about how expensive it's going to be (and the answer is fairly high), talk to Alper Altuntas for more information. We can run WW3 with no OBCs, but for the sake of explaining everything, we are going to do OBCs.

Given this criteria, I'm going to rip through a few steps.

In [ ]:
from pathlib import Path

from CrocoDash.grid import Grid
from CrocoDash.vgrid import VGrid
from CrocoDash.topo import Topo
from CrocoDash.case import Case


grid = Grid(
  resolution = 0.05, # in degrees
  xstart = 278.0, # min longitude in [0, 360]
  lenx = 5.0, # longitude extent in degrees
  ystart = 7.0, # min latitude in [-90, 90]
  leny = 5.0, # latitude extent in degrees
  name = "wave_fun_grid",
)

topo = Topo(
    grid = grid,
    min_depth = 9.5, # in meters
)

topo.set_from_dataset(    
    bathymetry_path = "<GEBCO_LOWRES>",
    longitude_coordinate_name="lon",
    latitude_coordinate_name="lat",
    vertical_coordinate_name="elevation"
)

topo.depth.plot()
vgrid  = VGrid.hyperbolic(
    nk = 75, # number of vertical levels
    depth = topo.max_depth,
    ratio=20.0 # target ratio of top to bottom layer thicknesses
)


In [ ]:
# CESM case (experiment) name
casename = "wave_fun_zone"

# CESM source root (Update this path accordingly!!!)
cesmroot ="<CESM>"

# Place where all your input files go 
inputdir = Path("<inputdir>") / casename
    
# CESM case directory
caseroot = Path("<casedir>") / casename


case = Case(
    cesmroot = cesmroot,
    caseroot = caseroot,
    inputdir = inputdir,
    ocn_grid = grid,
    ocn_vgrid = vgrid,
    ocn_topo = topo,
    project = '<PROJECT>',
    override = True,
    machine = "derecho",
    compset = "1850_DATM%NYF_SLND_SICE_MOM6%REGIONAL_SROF_SGLC_WW3" )


## WW3 Forcing

WW3 OBC Forcing also gets set up in a unique way. WW3 comes with a Fortran script to process "station-like" OBCs, and regrids the station data onto the WW3 grid. So all we do is prep "station-like" data. We take a slice of whatever data product we have (which in our case is ERA5 2D Wave Spectra), split the grid points into single point station files, create a file to list them (spec.list), set a few configuration parameters in a different file (bounc.nml) and place them in the input directory for waves (which is defined as a CESM xml variable). When you run `./preview_namelist` after `./case.build`, the WW3 part of the preview finds these boundary files and the compiled Fortran scripts and regrids them to your domain into a file called `nest.ww3`. 

ERA5 2d Wave Spectra is a **slow access product**, and requesting data can take **days** to get processed. **If you are interested in using ERA5 2d Wave Spectra, request it EARLY in the hacking part of this workshop**. You can request ERA5 2d Wave Spectra by running `case.configure_forcings()` and `case.process_forcings()` with the ERA5 2D Wave Spectra product (ERA5_WAVE_SPECTRA) and function name (get_era5_2d_spectra). You will need a login.

Because ERA5 2D Wave Spectra is slow, we can use a fake data product to sketch our our domain and make sure everything works first!

In [ ]:
case.configure_forcings(
    date_range=["2020-01-01 00:00:00", "2020-01-09 00:00:00"],
    function_name="get_glorys_data_from_rda",
    ww3_obc_product_name="reference_waves",              
    ww3_obc_function_name="get_reference_wave_spectra",           
)


In [ ]:
case.process_forcings()


A quick reminder that we are not creating a WW3 initial condition, so run your domain for a month or two!


Cool! Now, I would shake out the NTASKS & ROOPEs, build and submit! Based on how large your domain is, the preview namelist on submission make take a while (there's regridding going on after all)

Feel free to iterate with process forcings! You can run it on the command line with `crocodash process --caseroot YOURCASEROOT`!